<a href="https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS 5450 Final Project - UFO Shape Prediction**

# Part 1: Introduction

(todo)

# Part 2: Data Loading & Preprocessing

In [1]:
# !pip install reverse_geocoder

In [2]:
import pandas as pd
import numpy as np
import re
import reverse_geocoder as rg


from sklearn.neighbors import BallTree
import geopandas as gpd
from shapely.geometry import Point

## The UFO Sighting Dataset

###Information of The Raw Data

In [3]:
# Data loading via GitHub
github_url = 'https://raw.githubusercontent.com/jingyig16/ufo-sightings/refs/heads/main/ufo_sighting.csv'
ufo_df = pd.read_csv(github_url, low_memory=False)

In [4]:
ufo_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


In [5]:
ufo_df.columns = ufo_df.columns.str.strip().str.lower()
for i, col in enumerate(ufo_df.columns):
    print(f"{i+1}. {col}")

1. datetime
2. city
3. state
4. country
5. shape
6. duration (seconds)
7. duration (hours/min)
8. comments
9. date posted
10. latitude
11. longitude


In [6]:
len(ufo_df)

80332

In [7]:
ufo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              80332 non-null  object 
 1   city                  80332 non-null  object 
 2   state                 74535 non-null  object 
 3   country               70662 non-null  object 
 4   shape                 78400 non-null  object 
 5   duration (seconds)    80332 non-null  object 
 6   duration (hours/min)  80332 non-null  object 
 7   comments              80317 non-null  object 
 8   date posted           80332 non-null  object 
 9   latitude              80332 non-null  object 
 10  longitude             80332 non-null  float64
dtypes: float64(1), object(10)
memory usage: 6.7+ MB


### Data Processing

#### Handling Redundant and Invalid Data

Since the UFO `shape` is our target variable for prediction, we removed all records where this field was missing. Supervised learning models require a known label for each training example in order to compute loss and evaluate performance. Instances without a recorded shape cannot contribute to the learning process and would otherwise introduce noise or bias into the model. Therefore, rows with null values in the shape column were excluded from the dataset.

In [8]:
ufo_df = ufo_df.dropna(subset=['shape'])

In [9]:
len(ufo_df)

78400

In [10]:
ufo_cleaned = ufo_df.copy()

##### Drop Unused Columns

In [11]:
# Drop unused columns
ufo_cleaned = ufo_cleaned.drop(columns=["comments", "date posted","duration (hours/min)"])

The `duration (hours/min)` column was dropped because it duplicates the information in duration (seconds) but in inconsistent units. Keeping only the numeric duration simplifies analysis and ensures accurate quantitative modeling.

The `comments` column was removed since it contains unstructured text outside the scope of this study, which focuses on numerical and spatial patterns rather than linguistic content.

The `date posted` column was excluded because it reflects when a report was submitted, not when the sighting occurred. The true event timing is already captured in the `datetime` field, making date posted less meaningful for UFO pattern analysis.

#### Convert `datetime` to Datetime type

Converting the datetime column to datetime format ensures consistent and accurate time-based analysis and operations across the dataset.

We found that some records in `datetime` contained timestamps using "24:00", which is invalid in Python's datetime format. The value "24:00" represents midnight of the following day, but since most libraries only accept hours from 00 to 23, these entries caused conversion errors. To handle this, all "24:00" values were replaced with "00:00"

In [12]:
ufo_cleaned["datetime"] = (ufo_cleaned["datetime"].astype(str).apply(lambda x: re.sub(r"24:00", "00:00", x)))
ufo_cleaned["datetime"] = pd.to_datetime(ufo_cleaned["datetime"])

This small adjustment preserves the temporal meaning of the records while ensuring compatibility with pandas datetime operations.

#### Standardize `latitude` and `longitude`

In [13]:
def extract_numeric(value):
    """
    Extracts the first valid float-like number (with optional minus sign and decimal)
    from a string. Returns NaN if no valid number found.
    """
    if pd.isna(value):
        return None
    match = re.search(r"-?\d+\.?\d*", str(value))
    if match:
        return float(match.group())
    return None

In [14]:
ufo_cleaned["latitude"] = ufo_cleaned["latitude"].apply(extract_numeric)
ufo_cleaned["longitude"] = ufo_cleaned["longitude"].apply(extract_numeric)
# Convert to numeric just in case and drop invalid rows
ufo_cleaned["latitude"] = pd.to_numeric(ufo_cleaned["latitude"], errors="coerce")
ufo_cleaned["longitude"] = pd.to_numeric(ufo_cleaned["longitude"], errors="coerce")

In [15]:
ufo_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78400 entries, 0 to 80331
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            78400 non-null  datetime64[ns]
 1   city                78400 non-null  object        
 2   state               72742 non-null  object        
 3   country             69001 non-null  object        
 4   shape               78400 non-null  object        
 5   duration (seconds)  78400 non-null  object        
 6   latitude            78400 non-null  float64       
 7   longitude           78400 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 5.4+ MB


To ensure all coordinate information was standardized for downstream geospatial analysis, I implemented a custom cleaning pipeline using a regex-based function:*extract_numeric* to extract valid float-like numbers from potentially messy latitude and longitude entries. This approach handled irregular formats (e.g., values containing symbols, extra spaces, or text) and safely converted all coordinates to numeric types. After cleaning, both latitude and longitude columns contained 78,400 valid float64 values, confirming that all coordinate data were successfully retained and standardized.

#### Standardize `city`, `state`, and `country`

In [16]:
# Extract coordinates
coords = list(zip(ufo_cleaned["latitude"], ufo_cleaned["longitude"]))
# Batch reverse-geocode
results = rg.search(coords)
# Create geocoded DataFrame and align
geo_df = pd.DataFrame(results)
geo_df.index = ufo_cleaned.index
# Replace state & country with standardized values
ufo_cleaned["city"] = geo_df["name"]
ufo_cleaned["state"] = geo_df["admin1"]
ufo_cleaned["country"] = geo_df["cc"]

Loading formatted geocoded file...


We regenerated the `city`, `state`, and `country` columns directly from the latitude and longitude coordinates to achieve a consistent and standardized geographic representation across all records. The original text-based fields contained irregularities such as inconsistent capitalization, abbreviations, and mixed naming conventions, which made them unreliable for quantitative analysis or model training. In contrast, latitude and longitude are precise and continuous numerical values that can be used to objectively infer standardized location information through reverse geocoding. This ensures that entries with identical coordinates map to the same administrative regions, eliminating human errors and variations in naming.


In [17]:
ufo_cleaned

,datetime,city,state,country,shape,duration (seconds),latitude,longitude
0,1949-10-10 20:30:00,San Marcos,Texas,US,cylinder,2700,29.883056,-97.941111
1,1949-10-10 21:00:00,Lackland Air Force Base,Texas,US,light,7200,29.384210,-98.581082
2,1955-10-10 17:00:00,Blacon,England,GB,circle,20,53.200000,-2.916667
3,1956-10-10 21:00:00,Edna,Texas,US,circle,20,28.978333,-96.645833
4,1960-10-10 20:00:00,Kane'ohe,Hawaii,US,light,900,21.418056,-157.803611
...,...,...,...,...,...,...,...,...
80327,2013-09-09 21:15:00,Nashville,Tennessee,US,light,600,36.165833,-86.784444
80328,2013-09-09 22:00:00,Boise,Idaho,US,circle,1200,43.613611,-116.202500
80329,2013-09-09 22:00:00,Napa,California,US,other,1200,38.297222,-122.284444
80330,2013-09-09 22:20:00,Vienna,Virginia,US,circle,5,38.901111,-77.265556


In [18]:
ufo_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78400 entries, 0 to 80331
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            78400 non-null  datetime64[ns]
 1   city                78400 non-null  object        
 2   state               78400 non-null  object        
 3   country             78400 non-null  object        
 4   shape               78400 non-null  object        
 5   duration (seconds)  78400 non-null  object        
 6   latitude            78400 non-null  float64       
 7   longitude           78400 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 5.4+ MB


## City Features Dataset

In [19]:
city_github_url = 'https://raw.githubusercontent.com/jingyig16/ufo-sightings/refs/heads/main/new_dataset/city_features.csv'
city_df = pd.read_csv(city_github_url, low_memory=False)

In [20]:
city_df.head()

,city,city_ascii,lat,lng,country,iso2,iso3,admin_name,capital,population,id,wc_bio1,wc_bio4,wc_bio12,elevation_m,viirs_annual_ave
0,Tokyo,Tokyo,35.6870,139.7495,Japan,JP,JPN,Tōkyō,primary,37785000.0,1392685764,1.570417,769.836060,1431.0,33.0,28.124304
1,Jakarta,Jakarta,-6.1750,106.8275,Indonesia,ID,IDN,Jakarta,primary,33756000.0,1360771077,2.770000,37.598354,1958.0,6.0,32.622326
2,Delhi,Delhi,28.6100,77.2300,India,IN,IND,Delhi,admin,32226000.0,1356872604,2.517083,684.120056,730.0,214.0,67.107742
3,Guangzhou,Guangzhou,23.1300,113.2600,China,CN,CHN,Guangdong,admin,26940000.0,1156237133,2.232083,571.980041,1729.0,21.0,53.230152
4,Mumbai,Mumbai,19.0761,72.8775,India,IN,IND,Mahārāshtra,admin,24973000.0,1356226629,2.769167,182.356216,2159.0,6.0,37.921257


In [21]:
len(city_df)

48059

In [22]:
city_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48059 entries, 0 to 48058
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   city              48059 non-null  object 
 1   city_ascii        48057 non-null  object 
 2   lat               48059 non-null  float64
 3   lng               48059 non-null  float64
 4   country           48059 non-null  object 
 5   iso2              48026 non-null  object 
 6   iso3              48059 non-null  object 
 7   admin_name        47858 non-null  object 
 8   capital           15138 non-null  object 
 9   population        47808 non-null  float64
 10  id                48059 non-null  int64  
 11  wc_bio1           47484 non-null  float64
 12  wc_bio4           47484 non-null  float64
 13  wc_bio12          47484 non-null  float64
 14  elevation_m       47484 non-null  float64
 15  viirs_annual_ave  48055 non-null  float64
dtypes: float64(8), int64(1), object(7)
memor

In [23]:
city_cleaned_df = city_df.copy()

##### Drop Unused Columns

In [24]:
city_cleaned_df = city_cleaned_df.drop(['iso2', 'iso3', 'admin_name', 'id', 'capital'], axis=1)

The `iso2`and `iso3` columns were dropped because country information is already represented by the `country` field, while geographic location is fully captured by the `lat` and `lng` coordinates. Retaining multiple identifiers for the same attribute would introduce redundancy without adding analytical value.

The `admin_name`, `id` and `capital` columns were dropped because they contain administrative or metadata fields inherited from the original city dataset (e.g., province names, city IDs, or capital status) that are not relevant to the spatial analysis or the merging process with the UFO dataset. They were excluded to streamline the dataset and maintain focus on features directly related to location matching.

#### Standardize `latitude` and `longitude`



In [25]:
city_cleaned_df["lat"] = city_cleaned_df["lat"].apply(extract_numeric)
city_cleaned_df["lng"] = city_cleaned_df["lng"].apply(extract_numeric)

city_cleaned_df["lat"] = pd.to_numeric(city_cleaned_df["lat"], errors="coerce")
city_cleaned_df["lng"] = pd.to_numeric(city_cleaned_df["lng"], errors="coerce")

#### Standardize `city`, `state`, and `country`

In [26]:
# Extract coordinates
coords = list(zip(city_cleaned_df["lat"], city_cleaned_df["lng"]))
# Batch reverse-geocode
results = rg.search(coords)
# Create geocoded DataFrame and align
geo_df = pd.DataFrame(results)
geo_df.index = city_cleaned_df.index
# Replace state & country with standardized values
city_cleaned_df["city"] = geo_df["name"]
city_cleaned_df["state"] = geo_df["admin1"]
city_cleaned_df["country"] = geo_df["cc"]

Use the same method as how we processed `city`, `state` and `country` names in `ufo_cleaned` to ensure consistency.

In [27]:
city_cleaned_df.head()

,city,city_ascii,lat,lng,country,population,wc_bio1,wc_bio4,wc_bio12,elevation_m,viirs_annual_ave,state
0,Tokyo,Tokyo,35.6870,139.7495,JP,37785000.0,1.570417,769.836060,1431.0,33.0,28.124304,Tokyo
1,Jakarta,Jakarta,-6.1750,106.8275,ID,33756000.0,2.770000,37.598354,1958.0,6.0,32.622326,Jakarta Raya
2,New Delhi,Delhi,28.6100,77.2300,IN,32226000.0,2.517083,684.120056,730.0,214.0,67.107742,NCT
3,Guangzhou,Guangzhou,23.1300,113.2600,CN,26940000.0,2.232083,571.980041,1729.0,21.0,53.230152,Guangdong
4,Mumbai,Mumbai,19.0761,72.8775,IN,24973000.0,2.769167,182.356216,2159.0,6.0,37.921257,Maharashtra


## Join City Features with UFO Dataset

In [28]:
g_ufo   = gpd.GeoDataFrame(ufo_cleaned, geometry=gpd.points_from_xy(ufo_cleaned.longitude, ufo_cleaned.latitude), crs="EPSG:4326")
g_cities= gpd.GeoDataFrame(city_cleaned_df, geometry=gpd.points_from_xy(city_cleaned_df.lng, city_cleaned_df.lat), crs="EPSG:4326")

# Do in degrees or project first for metric threshold:
g_ufo_m  = g_ufo.to_crs(3857); g_cities_m = g_cities.to_crs(3857)
joined = (g_ufo_m.sjoin_nearest(g_cities_m, how='left', distance_col='dist_m')
        .reset_index()
        .sort_values(['index','dist_m'])
        # Keep only the closest city for each UFO record
        .drop_duplicates(subset=['index'], keep='first')
        .set_index('index'))

# Filter out far matches (e.g., >25 km)
# joined = joined[joined["dist_m"] <= 25000]

In [29]:
joined.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 78400 entries, 0 to 80331
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            78400 non-null  datetime64[ns]
 1   city_left           78400 non-null  object        
 2   state_left          78400 non-null  object        
 3   country_left        78400 non-null  object        
 4   shape               78400 non-null  object        
 5   duration (seconds)  78400 non-null  object        
 6   latitude            78400 non-null  float64       
 7   longitude           78400 non-null  float64       
 8   geometry            78400 non-null  geometry      
 9   index_right         78400 non-null  int64         
 10  city_right          78400 non-null  object        
 11  city_ascii          78399 non-null  object        
 12  lat                 78400 non-null  float64       
 13  lng                 78400 non-null  float64

We spatially join the cleaned UFO sightings data with the standardized city dataset to enrich each UFO record with corresponding city-level features such as population, elevation, and climate attributes.

We first convert both datasets into GeoDataFrames using `geopandas`, creating geometric points from their latitude and longitude values.

The `sjoin_nearest()` function performs a nearest-neighbor spatial join, which assigns each UFO record to the geographically closest city in the city dataset. This method is particularly useful because latitude and longitude values in different datasets rarely match exactly, even when referring to the same location. By matching based on spatial proximity, we can robustly link records that are near each other in real-world space rather than relying on potentially inconsistent textual identifiers like city names.

This approach ensures that every UFO observation in `ufo_cleaned` is matched to its most geographically appropriate city in `city_cleaned_df`, while maintaining consistency and minimizing false associations caused by rounding or naming discrepancies.

In [30]:
# Another method of joining
# R_EARTH_KM = 6371.0088

# def norm_txt(s):
#     return (s.astype(str)
#          .str.normalize('NFKD')
#          .str.encode('ascii', 'ignore')
#          .str.decode('ascii')
#          .str.lower()
#          .str.strip())

# # Step 1: Name-based join on standardized keys
# ufo = ufo_cleaned.copy()
# cities = city_cleaned_df.copy()

# # Standardize city, state, country across ufo and cities
# for df, (c,s,co) in [(ufo, ('city','state','country')),
#                      (cities, ('city','state','country'))]:
#     df['city_key'] = norm_txt(df[c])
#     df['state_key'] = norm_txt(df[s])
#     df['country_key'] = norm_txt(df[co])

# name_merge = ufo.merge(
#     cities,
#     on=['city_key','state_key','country_key'],
#     how='left',
#     suffixes=('','_city'))

# name_merge['match_method'] = np.where(name_merge['lat'].notna(), 'name', pd.NA)

# # Rows still unmatched after name join
# unmatched = name_merge[name_merge['lat'].isna()].copy()
# matched = name_merge[name_merge['lat'].notna()].copy()

# # Step 2: Nearest-neighbor geospatial join for the unmatched

# # Build BallTree from city coordinates (drop NaNs)
# cities_geo = cities.dropna(subset=['lat','lng']).copy()
# X = np.radians(cities_geo[['lat','lng']].to_numpy())

# # Query for unmatched UFO coords
# Y = np.radians(unmatched[['latitude','longitude']].to_numpy())
# tree = BallTree(X, metric='haversine')
# dist_rad, idx = tree.query(Y, k=1)
# dist_km = dist_rad[:,0] * R_EARTH_KM
# nearest_ix = idx[:,0]

# # Attach nearest city columns
# # add only columns not in ufo_cleaned
# cols_to_take = [c for c in city_cleaned_df.columns if c not in unmatched.columns]
# unmatched['nearest_dist_km'] = dist_km
# for col in cols_to_take:
#     unmatched[col] = cities_geo.iloc[nearest_ix][col].values
# unmatched['match_method'] = 'spatial'

# # # Optionally filter out far matches (e.g., >25 km)
# # MAX_KM = 25
# # nn.loc[nn['nearest_dist_km'] > MAX_KM, cities.columns] = np.nan

# # Step 3: Combine results
# merged_df = pd.concat([matched, unmatched], ignore_index=True)

# # Prefer name matches; otherwise use spatial
# merged_df.sort_values(by='match_method', ascending=True, inplace=True)
# merged_df = merged_df.sort_index().reset_index(drop=True)

# # Now have:
# # - Columns from UFO
# # - City attributes filled by name when possible, else by spatial nearest neighbor
# # - 'match_method' to audit the source and 'nearest_dist_km' to judge quality